In [13]:
import pandas as pd
import numpy as np

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import joblib

In [11]:
import warnings
warnings.filterwarnings('ignore')

In [4]:
rfm_scaled_df = pd.read_csv("rfm_for_model.csv")
rfm_original = pd.read_csv("rfm_original_values.csv")

In [5]:
rfm_log = np.log1p(rfm_original[['Recency', 'Frequency', 'Monetary']])

# Scaling (This creates a NumPy array)
scaler = StandardScaler()
rfm_scaled_array = scaler.fit_transform(rfm_log)

rfm_scaled_final = pd.DataFrame(rfm_scaled_array, 
                                index=rfm_original.index, 
                                columns=['Recency', 'Frequency', 'Monetary'])

rfm_scaled_final.dropna(inplace=True)

# Fit K-Means
kmeans = KMeans(n_clusters=4, n_init=10, random_state=42)
rfm_original['Cluster'] = kmeans.fit_predict(rfm_scaled_final)

### Prediction

In [9]:
def predict_new_customer(new_customer_data):
    log_data = np.log1p(new_customer_data)
    scaled_data = scaler.transform(log_data)
    cluster = kmeans.predict(scaled_data)

    return cluster[0]

In [12]:
print(predict_new_customer([[10, 2, 50]]))

3


### Export Model

In [14]:
import joblib
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.cluster import KMeans

In [15]:
# 1. Create a transformer for the log1p step
log_transformer = FunctionTransformer(np.log1p, validate=True)

# 2. Build the Pipeline
pipe = Pipeline([
    ('log', log_transformer),
    ('scaler', scaler),  # fitted scaler
    ('kmeans', kmeans)   # fitted kmeans model
])

# 3. Export the full pipeline
joblib.dump(pipe, 'rfm_pipeline.pkl')

['rfm_pipeline.pkl']